# Olist Brazilian E-commerce | Data Preparation & Exploration
Objective: Prepare the Olist dataset and document the evidence behind three business questions: (1) product sales trends, (2) delivery performance, and (3) delivery timing versus review scores.

Workflow: Python/Pandas for preparation and quality checks → MySQL for final KPIs and business queries → Power BI for the dashboard.

## 1. Load the source data

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path('data/raw')
FILES = {
    'orders': 'olist_orders_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

tables = {name: pd.read_csv(RAW / filename) for name, filename in FILES.items()}
orders = tables['orders']
order_items = tables['order_items']
reviews = tables['reviews']
payments = tables['payments']
products = tables['products']

pd.DataFrame({
    'table': list(tables),
    'rows': [len(df) for df in tables.values()],
    'columns': [len(df.columns) for df in tables.values()],
})

,table,rows,columns
0,orders,99441,8
1,customers,99441,5
2,order_items,112650,7
3,products,32951,9
4,payments,103886,5
5,reviews,99224,7
6,sellers,3095,4
7,geolocation,1000163,5
8,category_translation,71,2


## 2. Data quality and relationship checks

In [2]:
missing_summary = pd.concat(
    {name: df.isna().sum() for name, df in tables.items()},
    names=['table', 'column']
).rename('missing_count')
print(missing_summary[missing_summary.gt(0)].to_string())

quality_checks = {
    'duplicate_order_ids': orders['order_id'].duplicated().sum(),
    'duplicate_customer_ids': tables['customers']['customer_id'].duplicated().sum(),
    'orders_without_items': (~orders['order_id'].isin(order_items['order_id'])).sum(),
    'orders_without_payments': (~orders['order_id'].isin(payments['order_id'])).sum(),
    'orders_without_reviews': (~orders['order_id'].isin(reviews['order_id'])).sum(),
    'items_without_products': (~order_items['product_id'].isin(products['product_id'])).sum(),
    'items_without_sellers': (~order_items['seller_id'].isin(tables['sellers']['seller_id'])).sum(),
    'duplicate_geolocation_rows': tables['geolocation'].duplicated().sum(),
}
pd.Series(quality_checks, name='count').to_frame()


table     column                       
orders    order_approved_at                  160
          order_delivered_carrier_date      1783
          order_delivered_customer_date     2965
products  product_category_name              610
          product_name_lenght                610
          product_description_lenght         610
          product_photos_qty                 610
          product_weight_g                     2
          product_length_cm                    2
          product_height_cm                    2
          product_width_cm                     2
reviews   review_comment_title             87656
          review_comment_message           58247


,count
duplicate_order_ids,0
duplicate_customer_ids,0
orders_without_items,775
orders_without_payments,1
orders_without_reviews,768
items_without_products,0
items_without_sellers,0
duplicate_geolocation_rows,261831


## 3. Parse date columns and prepare limited cleaned copies

In [3]:
order_date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date'], errors='coerce'
)
for col in ['review_creation_date', 'review_answer_timestamp']:
    reviews[col] = pd.to_datetime(reviews[col], errors='coerce')

# Optional cleaned copies for future category/location analysis; not used in core KPIs.
products_clean = products.copy()
products_clean['product_category_name'] = (
    products_clean['product_category_name'].fillna('Unknown')
)
geolocation_clean = tables['geolocation'].drop_duplicates().copy()

print('Order date missing counts:')
print(orders[order_date_cols].isna().sum())
print('Geolocation rows after removing exact duplicates:', len(geolocation_clean))


Order date missing counts:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Geolocation rows after removing exact duplicates: 738332


## 4. Prepare an order-level analysis view

In [4]:
order_summary = (
    order_items.groupby('order_id', as_index=False)
    .agg(
        total_items=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
    )
)
orders_analysis = orders.merge(order_summary, on='order_id', how='left', validate='one_to_one')
orders_analysis['delivery_days'] = (
    orders_analysis['order_delivered_customer_date']
    - orders_analysis['order_purchase_timestamp']
).dt.total_seconds() / 86400

print('Order-level records:', len(orders_analysis))
print('Orders without item summaries:', orders_analysis['total_items'].isna().sum())


Order-level records: 99441
Orders without item summaries: 775


## 5. Quick exploratory cross-checks

In [5]:
orders_analysis['purchase_month'] = (
    orders_analysis['order_purchase_timestamp'].dt.to_period('M')
)
monthly_sales = (
    orders_analysis.groupby('purchase_month')['total_price'].sum(min_count=1)
)

delivered = orders_analysis.loc[
    orders_analysis['order_status'].eq('delivered')
    & orders_analysis['order_delivered_customer_date'].notna()
].copy()
delivered['is_late'] = (
    delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
)

print('Total recorded item sales (R$):', round(order_items['price'].sum(), 2))
print('Average delivery days:', round(delivered['delivery_days'].mean(), 2))
print('Late deliveries (%):', round(delivered['is_late'].mean() * 100, 2))
print('\nRecent monthly product sales:')
print(monthly_sales.tail(12))


Total recorded item sales (R$): 13591643.7
Average delivery days: 12.56
Late deliveries (%): 8.11

Recent monthly product sales:
purchase_month
2017-11    1010271.37
2017-12     743914.17
2018-01     950030.36
2018-02     844178.71
2018-03     983213.44
2018-04     996647.75
2018-05     996517.68
2018-06     865124.31
2018-07     895507.22
2018-08     854686.33
2018-09        145.00
2018-10           NaN
Freq: M, Name: total_price, dtype: float64
